## EMOTION_DETECTION

In [6]:
!python -m ipykernel install --user --name=tfenv --display-name "Python 3.10 (TensorFlow)"

Installed kernelspec tfenv in C:\Users\vv\AppData\Roaming\jupyter\kernels\tfenv


In [2]:
pip install tensorflow==2.15.0

  Using cached tensorflow-2.15.0-cp310-cp310-win_amd64.whl.metadata (3.6 kB)
  Using cached tensorflow_intel-2.15.0-cp310-cp310-win_amd64.whl.metadata (5.1 kB)
  Using cached ml_dtypes-0.2.0-cp310-cp310-win_amd64.whl.metadata (20 kB)
  Using cached keras-2.15.0-py3-none-any.whl.metadata (2.4 kB)
Using cached tensorflow-2.15.0-cp310-cp310-win_amd64.whl (2.1 kB)
Using cached tensorflow_intel-2.15.0-cp310-cp310-win_amd64.whl (300.9 MB)
Using cached keras-2.15.0-py3-none-any.whl (1.7 MB)
Using cached ml_dtypes-0.2.0-cp310-cp310-win_amd64.whl (938 kB)
   ---------------------------------------- 0.0/5.5 MB ? eta -:--:--
   ----- ---------------------------------- 0.8/5.5 MB 3.4 MB/s eta 0:00:02
   --------- ------------------------------ 1.3/5.5 MB 3.2 MB/s eta 0:00:02
   --------------- ------------------------ 2.1/5.5 MB 3.2 MB/s eta 0:00:02
   ------------------ --------------------- 2.6/5.5 MB 3.3 MB/s eta 0:00:01
   ------------------------ --------------- 3.4/5.5 MB 3.2 MB/s eta 0:00:0

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
deepface 0.0.93 requires opencv-python>=4.5.5.64, which is not installed.
retina-face 0.0.17 requires opencv-python>=3.4.4, which is not installed.

[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: C:\Users\vv\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip


In [1]:
import tensorflow as tf
import keras                     # Changed from: from tensorflow import keras
from keras import layers


In [2]:
import cv2
from deepface import DeepFace

# Fix raw string syntax
face_cascade = cv2.CascadeClassifier(r"C:\Users\vv\OneDrive\Desktop\PYTHON\haarcascade_frontalface_default.xml")
cap = cv2.VideoCapture(0)

# Frame counter to skip processing frames
frame_count = 0
skip_frames = 5
emotion = "Detecting..."

while True:
    ret, frame = cap.read()
    if not ret:
        break
        
    frame_count += 1
    gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    
    # Cascade needs gray, DeepFace needs standard BGR or RGB (convert directly from frame)
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    
    faces = face_cascade.detectMultiScale(
        gray_frame, 
        scaleFactor=1.1, 
        minNeighbors=5, 
        minSize=(30, 30)
    )
    
    for (x, y, w, h) in faces:
        # Draw bounding boxes every frame for smooth visuals
        cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 0, 255), 2)
        
        # Only run DeepFace analysis every X frames to prevent memory crash
        if frame_count % skip_frames == 0:
            face_roi = rgb_frame[y:y+h, x:x+w]
            try:
                result = DeepFace.analyze(face_roi, actions=['emotion'], enforce_detection=False)
                emotion = result[0]['dominant_emotion']
            except Exception as e:
                pass # Catch potential crop errors
                
        cv2.putText(frame, emotion, (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 0, 255), 2)
        
    cv2.imshow('Real-time Emotion Detection', frame)
    
    # 1ms wait is standard for live video; 27 is ESC key
    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()